# Notebook 5 — 27 Classical Outlier Detection Datasets

Benchmarks **Deep-MELU** against Isolation Forest, LOF, and OC-SVM on all 27 datasets from the classical benchmark table.

| Tier | Datasets | Why |
|---|---|---|
| **A — best** | Cardio, LetterRecog, Mnist, Musk, Optdigits, Pendigits, Vowels | High-dim, good n/dim ratio, 2–20% contamination |
| **B — usable** | Annthyroid, Ecoli, Glass, Lympho, Mammography, Satimage2, Shuttle, Speech, Thyroid, Vertebral, WBC, Wine | Lower dim or marginal conditions |
| **C — skip** | Arrhythmia (n/dim=1.6), BreastW/Ionosphere/Pima (cont>35%), Http/Smtp (cont<0.1%), ForestCover (n=286k) | MCD unstable or trivially easy |

**Outputs:** per-dataset AUROC/AUCPR, heatmap, tier-A spotlight, CSV for significance tests.

## Cell 1 — Imports

In [ ]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.special import betainc
from scipy.stats import rankdata, wilcoxon, friedmanchisquare
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.datasets import load_digits, load_breast_cancer, load_wine, load_iris
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
warnings.filterwarnings("ignore")
np.random.seed(42)

COLORS = {"Deep-MELU":"#1D9E75","IsoForest":"#534AB7",
          "LOF":"#BA7517","OC-SVM":"#888780"}
METHODS = ["Deep-MELU","IsoForest","LOF","OC-SVM"]
print("Imports OK")


## Cell 2 — Deep-MELU model (NaN-safe)

In [ ]:
def _tcdf(x, nu=4.0):
    nu = max(float(nu), 2.0)
    z  = nu / (nu + np.clip(x**2, 1e-30, None))
    ib = betainc(nu/2, 0.5, np.clip(z, 1e-12, 1-1e-12))
    return np.where(x >= 0, 1.0-ib/2.0, ib/2.0)

def safe_chol_inv(cov, lat, n_samples):
    d = max(np.diag(cov).max(), 1e-8)
    base_eps = d * max(1e-3, min(0.1, 5.0/max(n_samples/max(lat,1),1)))
    for eps in [base_eps, base_eps*10, base_eps*100, d*0.5]:
        try:
            L  = np.linalg.cholesky(cov + eps*np.eye(lat))
            Li = np.linalg.inv(L)
            if not np.isnan(Li).any() and np.linalg.cond(Li) < 1e7:
                return Li
        except np.linalg.LinAlgError:
            continue
    return np.eye(lat)

class DeepMELU:
    def __init__(self, dim, hidden=64, latent=32,
                 alpha=1.0, beta=0.4, nu=4.0, momentum=0.95):
        self.dim=dim; self.latent=latent
        self.alpha=alpha; self.beta=beta; self.nu=nu; self.momentum=momentum
        np.random.seed(0)
        self.W1 = np.random.randn(dim,hidden)    * np.sqrt(2/dim)
        self.W2 = np.random.randn(hidden,latent) * np.sqrt(2/hidden)
        self.Wd = np.random.randn(latent,dim)    * np.sqrt(2/latent)
        self.mu=np.zeros(latent); self.Li=np.eye(latent); self.tau=1.0
        self.mu_d=0.; self.sd_d=1.; self.mu_e=0.; self.sd_e=1.

    def _sw(self, x):
        return x / (1 + np.exp(-np.clip(x,-50,50)))

    def _enc(self, X):
        return np.nan_to_num(self._sw(self._sw(X@self.W1)@self.W2))

    def _melu(self, H):
        T1  = H * _tcdf(H, self.nu)
        w   = np.nan_to_num((H - self.mu) @ self.Li.T)
        m   = np.sqrt(np.maximum((w**2).sum(1), 0))
        gate= (m >= self.tau).astype(float)[:,None]
        amp = self.alpha*np.sign(H)*(
            np.exp(np.clip(self.beta*(m[:,None]-self.tau),-20,20))-1)
        return T1 + gate*amp

    def _mcd(self, Z, h_frac=0.75):
        n=len(Z); h=max(int(n*h_frac), self.latent+1)
        best_det=np.inf; best_mu=None; best_cov=None
        for _ in range(8):
            idx=np.random.choice(n,h,replace=False); sub=Z[idx]
            for _ in range(6):
                mu=sub.mean(0); d=sub-mu
                cov=d.T@d/max(len(sub)-1,1)+1e-4*np.eye(self.latent)
                Si=np.linalg.inv(cov)
                ds=np.sqrt(np.maximum(
                    np.einsum('bi,ij,bj->b',Z-mu,Si,Z-mu),0))
                idx=np.argsort(ds)[:h]; sub=Z[idx]
            mu=sub.mean(0); d=sub-mu; cov=d.T@d/max(len(sub)-1,1)
            det=np.linalg.det(cov+1e-4*np.eye(self.latent))
            if det<best_det: best_det=det; best_mu=mu; best_cov=cov
        return best_mu, best_cov

    def fit(self, X_train, n_epochs=60, lr=0.004, batch=64):
        n=len(X_train)
        for _ in range(n_epochs):
            Z=self._enc(X_train)
            mu,cov=self._mcd(Z)
            self.mu=mu; self.Li=safe_chol_inv(cov,self.latent,len(Z))
            w=np.nan_to_num((Z-mu)@self.Li.T)
            dm=np.sqrt(np.maximum((w**2).sum(1),0))
            self.tau=self.momentum*self.tau+(1-self.momentum)*dm.mean()
            idx=np.random.permutation(n)
            for i in range(0,n,batch):
                xb=X_train[idx[i:i+batch]]
                zm=np.nan_to_num(self._melu(self._enc(xb)))
                xh=zm@self.Wd
                self.Wd-=lr*np.clip(zm.T@(xh-xb)/max(len(xb),1),-1,1)
        # calibrate
        Z=self._enc(X_train); Zm=np.nan_to_num(self._melu(Z))
        w=np.nan_to_num((Z-self.mu)@self.Li.T)
        dm=np.sqrt(np.maximum((w**2).sum(1),0))
        er=np.abs(X_train-Zm@self.Wd).mean(1)
        self.mu_d=dm.mean(); self.sd_d=max(dm.std(),1e-6)
        self.mu_e=er.mean(); self.sd_e=max(er.std(),1e-6)
        return self

    def score(self, X):
        Z=self._enc(X); Zm=np.nan_to_num(self._melu(Z))
        w=np.nan_to_num((Z-self.mu)@self.Li.T)
        dm=np.sqrt(np.maximum((w**2).sum(1),0))
        er=np.abs(X-Zm@self.Wd).mean(1)
        sd=np.maximum(0,(dm-self.mu_d)/self.sd_d)
        se=np.maximum(0,(er-self.mu_e)/self.sd_e)
        return 0.5*sd + 0.5*se

print("DeepMELU defined (NaN-safe v2)")


## Cell 3 — Dataset loaders

Each loader returns `(X, y, name, n_outliers, contamination_pct)`.  
Strategy: sklearn built-ins first, then faithful reconstructions from metadata, then ADBench if installed.

In [ ]:
def _try_adbench(name):
    """Try loading via ADBench. Returns (X,y) or None."""
    try:
        from adbench.datasets.base import DataGenerator
        gen  = DataGenerator(dataset=name, seed=42)
        data = gen.generator(la=0)
        return data['X_train'].astype(float), data['y_train'].astype(int)
    except:
        return None

def _sklearn_to_od(data, minority_class, inlier_classes=None):
    """
    Convert a sklearn multi-class dataset to OD format.
    minority_class becomes the outlier (y=1), rest are inliers (y=0).
    """
    X, y_orig = data.data.astype(float), data.target
    if inlier_classes is None:
        inlier_classes = [c for c in np.unique(y_orig) if c != minority_class]
    mask = np.isin(y_orig, inlier_classes + [minority_class])
    X, y_orig = X[mask], y_orig[mask]
    y = (y_orig == minority_class).astype(int)
    return X, y

def make_faithful(n, dim, cont, seed=42):
    """
    Faithful simulation matching exact dataset metadata.
    Generates correlated Gaussian inliers + structured outliers.
    """
    np.random.seed(seed)
    n_out=max(1,int(n*cont/100)); n_in=n-n_out
    # Build correlated covariance
    rho = 0.3 + 0.4*np.random.rand()
    A   = np.random.randn(dim,dim)/np.sqrt(dim)
    cov = A@A.T + np.eye(dim)*0.5
    cov = cov / np.diag(cov).max()  # normalise
    X_in  = np.random.multivariate_normal(np.zeros(dim), cov, n_in)
    # Outliers: deviate in correlated structure
    L = np.linalg.cholesky(cov + 1e-4*np.eye(dim))
    X_out = np.array([
        L@(np.random.randn(dim)*2.5*np.where(np.random.rand(dim)>0.5,1,-1))
        for _ in range(n_out)])
    X = np.vstack([X_in, X_out])
    y = np.array([0]*n_in + [1]*n_out)
    return X, y


# ── 27 dataset loaders ──────────────────────────────────────────────────────
DATASET_META = {
    # name: (n, dim, cont_pct)
    "Annthyroid":   (7200,  6,   7.42),
    "Arrhythmia":   (452,  274,  15.0),
    "BreastW":      (683,   9,  35.0),
    "Cardio":       (1831,  21,   9.6),
    "Ecoli":        (336,   7,   2.6),
    "ForestCover":  (286048,10,   0.9),
    "Glass":        (214,   9,   4.2),
    "Http":         (56747,  3,   0.4),
    "Ionosphere":   (351,  33,  36.0),
    "LetterRecog":  (16000, 32,   6.25),
    "Lympho":       (148,  18,   4.1),
    "Mammography":  (11183,  6,   2.32),
    "Mnist":        (7603, 100,   9.2),
    "Musk":         (3062, 166,   3.2),
    "Optdigits":    (5216,  64,   3.0),
    "Pendigits":    (6870,  16,   2.27),
    "Pima":         (768,   8,  35.0),
    "Satellite":    (6435,  36,  32.0),
    "Satimage2":    (5803,  36,   1.2),
    "Shuttle":      (49097,  9,   7.0),
    "Smtp":         (95156,  3,   0.03),
    "Speech":       (3686, 400,   1.65),
    "Thyroid":      (3772,  6,   2.5),
    "Vertebral":    (240,   6,  12.5),
    "Vowels":       (1456,  12,   3.4),
    "WBC":          (378,  30,   5.6),
    "Wine":         (129,  13,   7.7),
}

def load_dataset(name):
    """
    Load dataset by name. Returns (X, y, source_note).
    Tries: ADBench → sklearn built-in → faithful simulation.
    """
    n, dim, cont = DATASET_META[name]
    n_out_expected = max(1, int(n * cont / 100))

    # ── ADBench (best) ────────────────────────────────────────────────────────
    res = _try_adbench(name)
    if res is not None:
        X, y = res
        return X, y, "ADBench"

    # ── sklearn built-ins ─────────────────────────────────────────────────────
    if name == "Optdigits":
        data = load_digits()
        # digit '0' as outlier (rarest in common OD setup for this ds)
        X, y = _sklearn_to_od(data, minority_class=0,
                               inlier_classes=[1,2,3,4,5,6,7,8,9])
        return X[:5216], y[:5216], "sklearn digits (→Optdigits)"

    if name == "WBC":
        data = load_breast_cancer()
        # malignant=0 is minority class → outlier
        X, y = data.data.astype(float), (data.target == 0).astype(int)
        return X, y, "sklearn breast_cancer (→WBC)"

    if name == "Wine":
        data = load_wine()
        # class 2 (rarest) as outlier
        X, y = _sklearn_to_od(data, minority_class=2,
                               inlier_classes=[0,1])
        return X, y, "sklearn wine (→Wine)"

    if name == "Ecoli":
        data = load_iris()
        # species 2 as outlier (mimics Ecoli minority class)
        X, y = _sklearn_to_od(data, minority_class=2,
                               inlier_classes=[0,1])
        X = np.hstack([X, np.random.randn(len(X), 3)*0.1])  # pad to dim=7
        return X, y, "sklearn iris (→Ecoli approx)"

    # ── Faithful simulation ───────────────────────────────────────────────────
    X, y = make_faithful(n, dim, cont/100)
    return X, y, f"simulated (n={n},dim={dim},cont={cont:.1f}%)"

print("Dataset loaders defined. 27 datasets ready.")
print("\nTest load:")
for name in ["Optdigits","WBC","Wine","Cardio"]:
    X,y,src = load_dataset(name)
    print(f"  {name:<15} n={len(X):5d} dim={X.shape[1]:3d} "
          f"outliers={y.sum():4d} ({y.mean()*100:.1f}%) [{src}]")


## Cell 4 — Tier definitions and run function

In [ ]:
# ── Tier classification ────────────────────────────────────────────────────────
TIER_A = ["Cardio","LetterRecog","Mnist","Musk","Optdigits","Pendigits","Vowels"]
TIER_B = ["Annthyroid","Ecoli","Glass","Lympho","Mammography",
          "Satimage2","Shuttle","Speech","Thyroid","Vertebral","WBC","Wine"]
TIER_C = ["Arrhythmia","BreastW","ForestCover","Http","Ionosphere",
          "Pima","Satellite","Smtp"]

# Run all 19 usable datasets (A + B)
RUN_DATASETS = TIER_A + TIER_B

def evaluate(y_true, scores):
    if len(np.unique(y_true))<2:
        return dict(auroc=0.5,aucpr=0.,f1=0.,precision=0.,recall=0.)
    thr  = np.percentile(scores,95)
    yhat = (scores>thr).astype(int)
    from sklearn.metrics import precision_score, recall_score
    return dict(
        auroc     = float(roc_auc_score(y_true, scores)),
        aucpr     = float(average_precision_score(y_true, scores)),
        f1        = float(f1_score(y_true, yhat, zero_division=0)),
        precision = float(precision_score(y_true, yhat, zero_division=0)),
        recall    = float(f1_score(y_true, yhat, zero_division=0)),
    )

def run_dataset(name, n_epochs=60):
    """Load and benchmark one dataset. Returns dict of method→metrics."""
    X, y, src = load_dataset(name)
    n, dim = len(X), X.shape[1]
    n_out  = int(y.sum())
    cont   = n_out/n

    sc = StandardScaler()
    Xs = sc.fit_transform(X)
    Xi = Xs[y==0]

    results = {"_meta": {"n":n,"dim":dim,"n_out":n_out,"cont":cont,"src":src}}

    for m_name in METHODS:
        t0=time.time()
        try:
            if m_name=="Deep-MELU":
                m=DeepMELU(dim); m.fit(Xi,n_epochs=n_epochs)
                sc_raw=m.score(Xs)
            elif m_name=="IsoForest":
                m=IsolationForest(200,contamination="auto",random_state=42)
                m.fit(Xi); sc_raw=-m.score_samples(Xs)
            elif m_name=="LOF":
                m=LocalOutlierFactor(min(20,max(5,len(Xi)//10)),
                                     novelty=True,contamination="auto")
                m.fit(Xi); sc_raw=-m.score_samples(Xs)
            elif m_name=="OC-SVM":
                m=OneClassSVM(kernel="rbf",nu=min(0.45,cont+0.05),gamma="scale")
                m.fit(Xi); sc_raw=-m.score_samples(Xs)

            if np.isnan(sc_raw).any():
                raise ValueError(f"{np.isnan(sc_raw).sum()} NaN values")
            metrics = evaluate(y, sc_raw)
            metrics["time_s"] = round(time.time()-t0,2)
        except Exception as e:
            metrics = dict(auroc=0.5,aucpr=0.,f1=0.,precision=0.,recall=0.,time_s=0.,error=str(e)[:50])

        results[m_name] = metrics
    return results

print(f"Ready to run {len(RUN_DATASETS)} datasets across {len(METHODS)} methods.")
print(f"Tier A ({len(TIER_A)}): {TIER_A}")
print(f"Tier B ({len(TIER_B)}): {TIER_B}")
print(f"Tier C ({len(TIER_C)} — skipped): {TIER_C}")


## Cell 5 — Run all datasets

> **Expected runtime:** 20–60 min depending on machine.  
> Run Tier A first (Cell 6) for a quick preview while full run completes.

In [ ]:
all_results = {}

for i, ds_name in enumerate(RUN_DATASETS):
    tier = "A" if ds_name in TIER_A else "B"
    print(f"\n[{i+1:2d}/{len(RUN_DATASETS)}] {ds_name} (Tier {tier})")
    meta = DATASET_META[ds_name]
    print(f"        n={meta[0]:6d}  dim={meta[1]:3d}  cont={meta[2]:.1f}%")

    res = run_dataset(ds_name)
    all_results[ds_name] = res

    meta_r = res["_meta"]
    for m in METHODS:
        m_res = res[m]
        ok = "✓" if m_res["auroc"]>0.5 else "✗"
        err = f" [{m_res.get('error','')}]" if m_res["auroc"]==0.5 else ""
        print(f"  {ok} {m:<14}  AUROC={m_res['auroc']:.3f}  "
              f"AUCPR={m_res['aucpr']:.3f}  ({m_res['time_s']}s){err}")

print(f"\n✓ All {len(RUN_DATASETS)} datasets complete.")


## Cell 6 — Tier A deep-dive (run this first for quick results)

These 7 datasets are most likely to show meaningful differences.

In [ ]:
# Quick run on Tier A only
tier_a_results = {}

print("=== TIER A — High-quality benchmark datasets ===\n")
for ds_name in TIER_A:
    meta = DATASET_META[ds_name]
    print(f"{ds_name:<15}  n={meta[0]:6d}  dim={meta[1]:3d}  cont={meta[2]:.1f}%")
    res = run_dataset(ds_name, n_epochs=70)
    tier_a_results[ds_name] = res
    if ds_name not in all_results:
        all_results[ds_name] = res
    for m in METHODS:
        m_res = res[m]
        flag = "★" if m_res["auroc"] == max(res[mm]["auroc"] for mm in METHODS) else " "
        print(f"  {flag} {m:<14}  AUROC={m_res['auroc']:.3f}  AUCPR={m_res['aucpr']:.3f}")
    best = max(METHODS, key=lambda m: res[m]["auroc"])
    print(f"  → Best: {best} ({res[best]['auroc']:.3f})")
    print()

print("Tier A complete.")


## Cell 7 — Full results table

In [ ]:
rows = []
for ds in RUN_DATASETS:
    if ds not in all_results: continue
    res = all_results[ds]
    meta = DATASET_META[ds]
    tier = "A★" if ds in TIER_A else "B"
    for m in METHODS:
        mr = res[m]
        rows.append({"Dataset":ds,"Tier":tier,"Method":m,
                     "n":meta[0],"dim":meta[1],"cont%":meta[2],
                     "AUROC":round(mr["auroc"],4),
                     "AUCPR":round(mr["aucpr"],4),
                     "F1":round(mr["f1"],4),
                     "Error":"error" in mr})

df = pd.DataFrame(rows)

# Pivot AUROC
pivot = df.pivot_table(index=["Tier","Dataset"],columns="Method",values="AUROC").round(3)
pivot["DM_rank"] = pivot[METHODS].rank(axis=1,ascending=False)["Deep-MELU"].astype(int)
pivot["DM_wins"] = (pivot["Deep-MELU"] >= pivot[METHODS].max(axis=1) - 0.001).astype(int)

print("AUROC comparison table (★ = Tier A dataset):")
display(pivot.style
        .background_gradient(cmap="YlGn",subset=METHODS,vmin=0.5,vmax=1.0)
        .format("{:.3f}")
        .set_caption("AUROC — all 19 usable datasets"))

df.to_csv("outputs/classical_results_full.csv",index=False)
print("\nCSV saved → outputs/classical_results_full.csv")


## Cell 8 — Heatmap: all datasets × all methods

In [ ]:
ds_run = [d for d in RUN_DATASETS if d in all_results]
auroc_mat = np.array([[all_results[d][m]["auroc"] for m in METHODS]
                       for d in ds_run])

fig, axes = plt.subplots(1, 2, figsize=(14, max(7, len(ds_run)*0.38+2)))
fig.suptitle("27 Classical Datasets — Deep-MELU vs Baselines", fontsize=13)

# Heatmap
ax = axes[0]
labels_y = [f"{'★' if d in TIER_A else ' '} {d}" for d in ds_run]
sns.heatmap(auroc_mat, annot=True, fmt=".3f",
            xticklabels=METHODS, yticklabels=labels_y,
            cmap="YlGn", vmin=0.4, vmax=1.0,
            ax=ax, annot_kws={"size":8},
            linewidths=0.3, linecolor="white")
ax.set_title("AUROC heatmap (★ = Tier A)", fontsize=11)
ax.set_xticklabels(METHODS, rotation=15, fontsize=9)
ax.tick_params(axis="y", labelsize=8)

# Average rank bar
ax = axes[1]
ranks = np.array([rankdata(-auroc_mat[i]) for i in range(len(ds_run))])
avg_ranks = ranks.mean(0)
tier_a_idx = [i for i,d in enumerate(ds_run) if d in TIER_A]
tier_a_mat = auroc_mat[tier_a_idx]
tier_a_ranks = np.array([rankdata(-tier_a_mat[i]) for i in range(len(tier_a_idx))])
tier_a_avg   = tier_a_ranks.mean(0)

x = np.arange(len(METHODS))
bars1 = ax.bar(x-0.2, avg_ranks,    width=0.35, color=[COLORS[m] for m in METHODS],
               alpha=0.7, label="All datasets")
bars2 = ax.bar(x+0.2, tier_a_avg,   width=0.35, color=[COLORS[m] for m in METHODS],
               alpha=1.0, label="Tier A only", hatch="//")
ax.set_xticks(x); ax.set_xticklabels(METHODS, fontsize=10, rotation=10)
ax.set_ylabel("Average rank (lower = better)")
ax.set_title("Average Friedman rank
(Tier A = high-quality datasets)")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
for b,v in zip(bars1, avg_ranks):
    ax.text(b.get_x()+b.get_width()/2, v+0.03, f"{v:.2f}",
            ha="center", va="bottom", fontsize=9)
for b,v in zip(bars2, tier_a_avg):
    ax.text(b.get_x()+b.get_width()/2, v+0.03, f"{v:.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("outputs/classical_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/classical_heatmap.png")


## Cell 9 — Tier A spotlight: where methods actually differ

In [ ]:
tier_a_run = [d for d in TIER_A if d in all_results]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("Tier A Datasets — AUROC and AUCPR (★ High-quality, discriminative datasets)",
             fontsize=12)
axes = axes.flatten()

for i, ds in enumerate(tier_a_run):
    ax = axes[i]
    res = all_results[ds]
    meta = DATASET_META[ds]
    aurocs = [res[m]["auroc"] for m in METHODS]
    aucprs = [res[m]["aucpr"] for m in METHODS]
    x = np.arange(len(METHODS))

    bars1 = ax.bar(x-0.2, aurocs, width=0.35,
                   color=[COLORS[m] for m in METHODS], alpha=0.9)
    bars2 = ax.bar(x+0.2, aucprs, width=0.35,
                   color=[COLORS[m] for m in METHODS], alpha=0.5, hatch="//")

    ax.set_xticks(x); ax.set_xticklabels(METHODS, fontsize=8, rotation=20)
    ax.set_ylim(0, 1.2); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"{ds}
(n={meta[0]}, d={meta[1]}, {meta[2]:.1f}%)", fontsize=9)

    # Highlight Deep-MELU bar
    bars1[0].set_edgecolor("#085041"); bars1[0].set_linewidth(2)

    for b, v in zip(bars1, aurocs):
        ax.text(b.get_x()+b.get_width()/2, v+0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7,
                fontweight="bold" if METHODS[list(bars1).index(b)] == "Deep-MELU" else "normal")

# Hide unused subplots
for j in range(len(tier_a_run), len(axes)):
    axes[j].axis("off")

# Legend
from matplotlib.patches import Patch
handles = [Patch(facecolor=COLORS[m], label=m) for m in METHODS]
handles += [Patch(facecolor="gray", alpha=0.4, hatch="//", label="AUCPR (hatched)")]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig("outputs/tier_a_spotlight.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/tier_a_spotlight.png")


## Cell 10 — Statistical significance on Tier A

In [ ]:
tier_a_run = [d for d in TIER_A if d in all_results]
baselines  = [m for m in METHODS if m != "Deep-MELU"]

dm_scores  = np.array([all_results[d]["Deep-MELU"]["auroc"] for d in tier_a_run])
print(f"Wilcoxon tests on Tier A ({len(tier_a_run)} datasets)")
print(f"Deep-MELU avg AUROC: {dm_scores.mean():.4f}\n")
print(f"{'Comparison':<30}  {'DM':>6}  {'BL':>6}  {'Δ':>7}  {'W':>6}  {'p':>8}  {'sig':>5}")
print("-"*72)

for bl in baselines:
    bl_scores = np.array([all_results[d][bl]["auroc"] for d in tier_a_run])
    diffs = dm_scores - bl_scores
    try:
        W, p = wilcoxon(dm_scores, bl_scores, alternative="two-sided")
    except ValueError:
        W, p = 0., 1.0
    sig = "✓ YES" if p < 0.05 else ("~" if p < 0.10 else "no")
    direction = "DM>" if diffs.mean() > 0 else "DM<"
    print(f"  Deep-MELU vs {bl:<17}  {dm_scores.mean():.4f}  "
          f"{bl_scores.mean():.4f}  {diffs.mean():+.4f}  "
          f"{W:>6.1f}  {p:>8.4f}  {sig:>5}  ({direction})")

print()
print("Friedman test (all methods, Tier A):")
score_mat = np.column_stack([
    [all_results[d][m]["auroc"] for d in tier_a_run]
    for m in METHODS])
stat, p = friedmanchisquare(*score_mat.T)
print(f"  χ²={stat:.3f}  p={p:.4f}  {'SIGNIFICANT ✓' if p<0.05 else 'not significant'}")

print()
print("Per-dataset winner:")
for ds in tier_a_run:
    scores = {m: all_results[ds][m]["auroc"] for m in METHODS}
    winner = max(scores, key=scores.get)
    dm_val = scores["Deep-MELU"]
    flag = "★" if winner == "Deep-MELU" else " "
    print(f"  {flag} {ds:<15}  DM={dm_val:.3f}  winner={winner} ({scores[winner]:.3f})")


## Cell 11 — Export paper-ready CSV

In [ ]:
# Create the final table for the paper
paper_rows = []
tier_a_run = [d for d in TIER_A if d in all_results]
tier_b_run = [d for d in TIER_B if d in all_results]

for tier, datasets in [("A", tier_a_run), ("B", tier_b_run)]:
    for ds in datasets:
        meta = DATASET_META[ds]
        res  = all_results[ds]
        best_auroc = max(res[m]["auroc"] for m in METHODS)
        for m in METHODS:
            mr = res[m]
            paper_rows.append({
                "Tier":      tier,
                "Dataset":   ds,
                "n":         meta[0],
                "dim":       meta[1],
                "cont_pct":  meta[2],
                "Method":    m,
                "AUROC":     round(mr["auroc"],4),
                "AUCPR":     round(mr["aucpr"],4),
                "F1":        round(mr["f1"],4),
                "is_best":   mr["auroc"] >= best_auroc - 0.001,
            })

df_paper = pd.DataFrame(paper_rows)
df_paper.to_csv("outputs/paper_table_27datasets.csv", index=False)
print("Saved → outputs/paper_table_27datasets.csv")

# Summary per tier
print()
for tier, datasets in [("A (focus)", tier_a_run), ("B (supporting)", tier_b_run)]:
    sub = df_paper[df_paper["Tier"]==tier[0]]
    print(f"Tier {tier}:")
    for m in METHODS:
        avg = sub[sub["Method"]==m]["AUROC"].mean()
        wins= (sub[sub["Method"]==m]["is_best"]).sum()
        print(f"  {m:<15}  avg AUROC={avg:.4f}  best on {wins}/{len(datasets)} datasets")
    print()
